# 02 Transforms, Kinematics, And IK

本章把已知物体位姿转成机器人关节运动。主线是：transforms -> forward kinematics -> Jacobian -> differential IK。

![Transforms, kinematics, and IK visual map](assets/figures/02_transforms_kinematics_ik.png)

## Learning Objectives

- Use frame names to compose 2D rigid transforms without losing direction.
- Connect forward kinematics to the planar manipulator Jacobian.
- Explain why differential IK is local, iterative, and target-limited by reachability.

## Checkpoint

- Given `A_from_B` and `B_from_C`, write the composed transform in the correct order.
- Predict which Jacobian column changes most when a joint is near the base.
- Run the IK cell and explain the final error in task-space terms.

## Practice Task

Modify the IK target twice: one reachable point and one unreachable point. Record the final error and explain the difference in one paragraph.

## Concept Map

![Colab concept image](assets/colab/02_transforms_kinematics_ik_concept.png)

**Concept image.** Frames define where objects are; kinematics and the Jacobian turn frame error into joint updates.

In [ ]:
from pathlib import Path
import sys

COLAB_REPO_URL = "https://github.com/Hollis36/robotic-manipulation-learning.git"

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    in_colab = "google.colab" in sys.modules
    if in_colab:
        import subprocess

        PROJECT_ROOT = Path("/content/robotic-manipulation-learning")
        if not PROJECT_ROOT.exists():
            # Equivalent command: git clone --depth 1 <repo> <target>
            subprocess.run(["git", "clone", "--depth", "1", COLAB_REPO_URL, str(PROJECT_ROOT)], check=True)
    else:
        PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

PROJECT_ROOT


## Spatial Transform Composition

命名规则：`A_from_B @ B_from_C = A_from_C`。

In [ ]:
import numpy as np
from rml.transforms import apply_transform, compose, make_transform

world_from_gripper = make_transform(np.pi / 2, [0.5, 0.2])
gripper_from_brick = make_transform(0.0, [0.1, 0.0])
world_from_brick = compose(world_from_gripper, gripper_from_brick)
brick_origin_world = apply_transform(world_from_brick, np.array([[0.0, 0.0]]))[0]
np.round(world_from_brick, 3), np.round(brick_origin_world, 3)

## Forward Kinematics And Jacobian

FK 回答“现在末端在哪里”，Jacobian 回答“每个关节微小变化如何影响末端”。

In [ ]:
from rml.kinematics import planar_forward_kinematics, planar_jacobian

q = np.array([0.4, -0.2, 0.3])
links = np.array([0.7, 0.5, 0.2])
position = planar_forward_kinematics(q, links)
jacobian = planar_jacobian(q, links)
np.round(position, 3), np.round(jacobian, 3)

## Differential IK

微分 IK 使用局部线性近似，一步一步减小 task-space error。

In [ ]:
from rml.differential_ik import solve_planar_position_ik

target = np.array([1.3, 0.7])
initial_q = np.array([0.1, 0.1, -0.2])
result = solve_planar_position_ik(initial_q, target, np.array([1.0, 0.8, 0.4]), steps=200, damping=0.05, gain=0.6)
final_position = planar_forward_kinematics(result.joint_angles, np.array([1.0, 0.8, 0.4]))
np.round(result.joint_angles, 3), np.round(final_position, 3), result.error_norm

## Result Figure

Plot the final planar arm, the desired target, and the remaining end-effector error.

The figure below is generated from the values computed in this notebook. Treat it as evidence from the code, not as a decorative illustration.

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({
    "figure.figsize": (7, 4.2),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
})
import numpy as np

links = np.array([1.0, 0.8, 0.4])
cumulative_angles = np.cumsum(result.joint_angles)
segments = np.column_stack((links * np.cos(cumulative_angles), links * np.sin(cumulative_angles)))
points = np.vstack((np.array([[0.0, 0.0]]), np.cumsum(segments, axis=0)))
fig, ax = plt.subplots()
ax.plot(points[:, 0], points[:, 1], '-o', lw=4, label='final arm')
ax.scatter([target[0]], [target[1]], marker='*', s=180, label='target')
ax.scatter([final_position[0]], [final_position[1]], s=90, label='end effector')
ax.plot([final_position[0], target[0]], [final_position[1], target[1]], '--', label=f'error={result.error_norm:.3f}')
ax.set_aspect('equal', adjustable='box')
ax.set_xlabel('x position')
ax.set_ylabel('y position')
ax.legend(frameon=False)
plt.show()

## Parameter Experiment

The next cell is marked with `COLAB_PARAMETER_EXPERIMENT` so it is easy to find in Colab. Change `target_options` and compare how final error changes for reachable and unreachable targets.

In [ ]:
# COLAB_PARAMETER_EXPERIMENT
target_options = [np.array([1.3, 0.7]), np.array([2.0, 0.1]), np.array([3.0, 0.0])]
for candidate in target_options:
    trial = solve_planar_position_ik(initial_q, candidate, np.array([1.0, 0.8, 0.4]), steps=200, damping=0.05, gain=0.6)
    print(candidate.tolist(), 'final_error=', round(trial.error_norm, 4))

## Reflection Prompt

为什么不可达目标的误差不会归零？用 link length 总和和 task-space error 解释。

Exercise: 把 target 改成 `[3.0, 0.0]`，解释为什么误差无法真正变成 0。